# Lesson 02 - Exploring Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** is a unified framework for building AI agents. It provides a clean, composable architecture with four core building blocks:

- **Client** – connects to an AI model endpoint and handles communication
- **Agent** – wraps a client with instructions and tool definitions
- **Tools** – extend agent capabilities with custom functions the model can call
- **Session** – maintains conversation history for multi-turn interactions

In this lesson, we'll build a **travel booking agent** that checks destination availability using these concepts.

## Setup

In [19]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects agent-framework-foundry azure-identity -U -q

In [20]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
# from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from agent_framework import Agent

## Understanding the Agent Framework Architecture

The Microsoft Agent Framework follows a layered architecture:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – An `AzureAIProjectAgentProvider` connects to an Azure OpenAI deployment. It handles authentication, request formatting, and response parsing.
2. **Agent** – Created from the client via `provider.create_agent()`, the agent combines model access with instructions (system prompt) and tools.
3. **Tools** – Python functions decorated with `@tool` that the agent can invoke to perform actions or retrieve data.
4. **Session** – An `AgentSession` object (created via `agent.create_session()`) that stores conversation history, enabling multi-turn dialogue where the agent remembers prior context.

Let's build each layer step by step.

## Client Setup: Foundry vs GitHub Models

You can use **either**:
- **Foundry** (Azure AI Foundry) – requires `FOUNDRY_PROJECT_ENDPOINT` and model deployment
- **GitHub Models** (free) – requires `GITHUB_TOKEN` env var

Select one approach below.

In [21]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Option 1: Use Foundry (if you have a project + model deployment)
try:
    use_github = USE_GITHUB_MODELS if 'USE_GITHUB_MODELS' in globals() else False
    
    if use_github:
        print("↳ USE_GITHUB_MODELS=True, skipping Foundry...")
        raise ValueError("Forced to use GitHub Models")
    
    foundry_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    foundry_model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
    
    if foundry_endpoint and foundry_model:
        print("✓ Using Foundry endpoint + model")
        client = FoundryChatClient(
            credential=AzureCliCredential(),
            project_endpoint=foundry_endpoint,
            model=foundry_model,
        )
        print(f"  Model: {foundry_model}")
    else:
        raise ValueError("Foundry env vars missing")
except Exception as e:
    print(f"✗ Foundry setup failed ({e}), falling back to GitHub Models...")
    
    # Option 2: Use GitHub Models (free, requires GITHUB_TOKEN)
    from openai import AsyncOpenAI
    
    github_token = os.getenv("GITHUB_TOKEN")
    if not github_token:
        raise ValueError("GITHUB_TOKEN env var not found. Set it to use GitHub Models.")
    
    print("✓ Using GitHub Models")
    client = AsyncOpenAI(
        api_key=github_token,
        base_url="https://models.inference.ai.azure.com",
    )
    # For GitHub Models, we'll use this model name
    # Options: gpt-4o, gpt-4o-mini, claude-3.5-sonnet, phi-3, llama-3.1, mistral-large, etc.
    GITHUB_MODEL = "gpt-4o"
    print(f"  Model: {GITHUB_MODEL}")


↳ USE_GITHUB_MODELS=True, skipping Foundry...
✗ Foundry setup failed (Forced to use GitHub Models), falling back to GitHub Models...
✓ Using GitHub Models
  Model: gpt-4o


In [22]:
# (Optional) Force use GitHub Models instead of Foundry
# Set to True to bypass Foundry (useful if gpt-oss-120b doesn't work on your deployment)
# Set to False to use Foundry (default)

USE_GITHUB_MODELS = True  # ← Change to True to test with GitHub Models, False for Foundry

if USE_GITHUB_MODELS:
    print("⚠️  Will use GitHub Models (bypassing Foundry)")
else:
    print("ℹ️  Will use Foundry (if available)")


⚠️  Will use GitHub Models (bypassing Foundry)


## Adding Tools with the @tool Decorator

Tools let agents take actions beyond generating text. The `@tool` decorator converts a regular Python function into something the agent can call.

Key points:
- Use `Annotated[type, "description"]` so the model understands each parameter.
- The docstring becomes the tool description the model sees.
- `approval_mode="never_require"` means the tool runs automatically without user confirmation.

In [23]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

# Có thể bổ sung thêm hàm này để turn 1 có thể gọi
# @tool(approval_mode="never_require")
# def list_destinations() -> str:
#     """List all vacation destinations that can be checked for availability."""
#     return "Available destinations to check: Barcelona, Tokyo, Cape Town, Vancouver, Dubai."


## Creating an Agent with Tools

Now we combine the client, instructions, and tools into an agent. The `instructions` act as the system prompt — they define the agent's persona and behaviour.

In [24]:
# Create agent from the client
# The agent wraps the client with instructions and tools

from agent_framework import Agent

if isinstance(client, FoundryChatClient):
    # Foundry path: use Agent directly
    agent = Agent(
        client=client,
        instructions=(
            "You are a travel booking agent. Help users check destination availability "
            "and make recommendations. Always check availability before recommending a destination."
        ),
        name="TravelAvailabilityAgent",
        tools=[check_destination_availability],
    )
    print("✓ Agent created with Foundry client")
else:
    # GitHub Models path: for now, we'll use the client directly in a simpler way
    # (agent_framework's Agent class is designed for FoundryChatClient specifically)
    print("ℹ Using GitHub Models client directly (simplified mode without agent framework wrapper)")
    agent = None  # We'll handle GitHub Models differently below


ℹ Using GitHub Models client directly (simplified mode without agent framework wrapper)


## Multi-Turn Conversations with Sessions

An `AgentSession` (created via `agent.create_session()`) keeps track of all messages in a conversation. By passing the same session to each `agent.run()` call, the agent has access to the full conversation history and can refer back to earlier messages.

We pass `tools=[check_destination_availability]` so the agent can call our availability checker during each turn.

In [25]:
# Turn 1: Ask about available destinations

if isinstance(client, FoundryChatClient):
    # Foundry path
    session = agent.create_session()
    response = await agent.run(
        messages="Which destinations do you have available?",
        session=session
    )
    print(f"Agent: {response.text}")
else:
    # GitHub Models path: use OpenAI client directly
    # Initialize conversation history for multi-turn support
    conversation_history = [
        {
            "role": "system",
            "content": "You are a travel booking agent. Help users check destination availability and make recommendations."
        }
    ]
    
    print("Calling GitHub Models API...")
    user_message_1 = "Which destinations do you have available? (Barcelona, Tokyo, Cape Town, Vancouver, Dubai)"
    conversation_history.append({"role": "user", "content": user_message_1})
    
    response = await client.chat.completions.create(
        model=GITHUB_MODEL,
        messages=conversation_history,
        temperature=1,
        max_tokens=1024,
    )
    
    assistant_response = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": assistant_response})
    print(f"Agent: {assistant_response}")


Calling GitHub Models API...
Agent: Great choices! Let me check availability for you at **Barcelona, Tokyo, Cape Town, Vancouver, and Dubai.**  

### Availability Status:
1. **Barcelona, Spain**: ✅ Available  
   - Best time to visit: April to June, September to October  
   - Highlights: Sagrada Família, Park Güell, Gothic Quarter, beaches  

2. **Tokyo, Japan**: ✅ Available  
   - Best time to visit: March to April (cherry blossom season) or October to November (autumn foliage)  
   - Highlights: Shibuya Crossing, Asakusa, Tokyo Skytree, Tsukiji Market  

3. **Cape Town, South Africa**: ✅ Available  
   - Best time to visit: November to March (summer) or September to November (spring)  
   - Highlights: Table Mountain, Cape Point, Robben Island, wine tours  

4. **Vancouver, Canada**: ✅ Available  
   - Best time to visit: March to May or September to November  
   - Highlights: Stanley Park, Granville Island, Capilano Suspension Bridge, mountain hiking  

5. **Dubai, UAE**: ✅ Availa

In [26]:
# Debug: Inspect conversation history (only for GitHub Models)
if not isinstance(client, FoundryChatClient):
    print("📋 Conversation History After Turn 1:")
    print(f"   Total messages: {len(conversation_history)}")
    for i, msg in enumerate(conversation_history):
        role = msg["role"].upper()
        content = msg["content"][:80] + "..." if len(msg["content"]) > 80 else msg["content"]
        print(f"   [{i}] {role}: {content}")
else:
    print("✓ Foundry uses AgentSession (multi-turn built-in)")


📋 Conversation History After Turn 1:
   Total messages: 3
   [0] SYSTEM: You are a travel booking agent. Help users check destination availability and ma...
   [1] USER: Which destinations do you have available? (Barcelona, Tokyo, Cape Town, Vancouve...
   [2] ASSISTANT: Great choices! Let me check availability for you at **Barcelona, Tokyo, Cape Tow...


In [27]:
# Turn 2: Follow-up question (multi-turn conversation)

if isinstance(client, FoundryChatClient):
    # Foundry path
    response = await agent.run(
        "I'd like to go somewhere warm. What's available?",
        session=session,
    )
    print(f"Agent: {response.text}")
else:
    # GitHub Models path: continue with conversation history
    print("Calling GitHub Models API (with Turn 1 history)...")
    user_message_2 = "I'd like to go somewhere warm. What's available?"
    conversation_history.append({"role": "user", "content": user_message_2})
    
    response = await client.chat.completions.create(
        model=GITHUB_MODEL,
        messages=conversation_history,
        temperature=1,
        max_tokens=1024,
    )
    
    assistant_response = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": assistant_response})
    print(f"Agent: {assistant_response}")


Calling GitHub Models API (with Turn 1 history)...
Agent: If you're looking for a warm destination, here are the top picks from your list, based on current availability:

### Warm Destination Recommendations:
1. **Dubai, UAE**  
   - **Weather**: Warm year-round. Temperatures average 25-35°C (77-95°F) in the cooler months (November-March), perfect for outdoor exploration.  
   - **Highlights**: Luxury shopping, desert safaris, beaches, and world-famous landmarks like the Burj Khalifa.  
   - **Recommendation**: Ideal for a luxurious getaway with a mix of culture, adventure, and relaxation.

2. **Cape Town, South Africa**  
   - **Weather**: Visit between November and March for summer temperatures of 20-30°C (68-86°F).  
   - **Highlights**: Gorgeous beaches, vineyards, wildlife safaris, hiking, and stunning views from Table Mountain.  
   - **Recommendation**: Perfect for a mix of natural beauty, adventure, and cultural experiences.

3. **Barcelona, Spain** *(spring or summer)*  
   - 

In [28]:
# Validation: Verify multi-turn is working (only for GitHub Models)
if not isinstance(client, FoundryChatClient):
    print("\n✅ Multi-Turn Verification:")
    print(f"   Total messages in history: {len(conversation_history)}")
    print(f"   Expected: 5 (1 system + 2 user + 2 assistant)")
    
    has_turn1_user = any("destinations do you have available" in msg["content"] for msg in conversation_history if msg["role"] == "user")
    has_turn1_assistant = len([m for m in conversation_history if m["role"] == "assistant"]) >= 1
    has_turn2_user = any("somewhere warm" in msg["content"] for msg in conversation_history if msg["role"] == "user")
    
    if has_turn1_user and has_turn1_assistant and has_turn2_user:
        print("   ✓ Turn 1 user question in history")
        print("   ✓ Turn 1 agent response in history")
        print("   ✓ Turn 2 user question in history")
        print("\n   🎉 Multi-turn is WORKING! Agent can see full conversation.")
    else:
        print("   ⚠ Some messages missing from history")



✅ Multi-Turn Verification:
   Total messages in history: 5
   Expected: 5 (1 system + 2 user + 2 assistant)
   ✓ Turn 1 user question in history
   ✓ Turn 1 agent response in history
   ✓ Turn 2 user question in history

   🎉 Multi-turn is WORKING! Agent can see full conversation.


## Summary

In this lesson you explored the four pillars of the Microsoft Agent Framework:

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` connects to Azure OpenAI with credential-based auth |
| **Agent** | `provider.create_agent()` bundles a model connection with instructions and a name |
| **Tools** | The `@tool` decorator exposes Python functions for the agent to call |
| **Session** | `agent.create_session()` maintains conversation history across multiple turns |

These building blocks compose together to create agents that can hold natural conversations, call external functions, and maintain context — the foundation for more advanced agentic patterns in later lessons.